In [ ]:
%pip install tweety-ns
%pip install pyotp
%pip install pocketbase
%pip install loguru

In [ ]:
import json
import math
from enum import Enum
from pathlib import Path
from pprint import pprint
from typing import Literal, Optional

import pyotp
from google.colab import drive, userdata
from loguru import logger
from pocketbase import PocketBase
from pocketbase.errors import ClientResponseError
from pocketbase.models import Record
from pydantic import BaseModel
from tweety import TwitterAsync
from tweety.filters import SearchFilters
from tweety.types import SelfThread
from tweety.types import Tweet as TweetyTweet

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')

In [ ]:
class Settings(Enum):
    """Settings class to hold environment variables."""
    X_USERNAME = userdata.get('X_USERNAME')
    X_PASSWORD = userdata.get("X_PASSWORD")
    PYOTP = pyotp.TOTP(userdata.get("X_TOTP"))
    X_TOTP = PYOTP.now()

    POCKETBASE_EMAIL = userdata.get("POCKETBASE_EMAIL")
    POCKETBASE_PASSWORD = userdata.get("POCKETBASE_PASSWORD")
    POCKETBASE_URL = userdata.get("POCKETBASE_URL")

In [ ]:
assert Settings.X_USERNAME, "X_USERNAME is not set"
assert Settings.X_PASSWORD, "X_PASSWORD is not set"
assert Settings.PYOTP, "X_TOTP is not set"
assert Settings.POCKETBASE_EMAIL, "POCKETBASE_EMAIL is not set"
assert Settings.POCKETBASE_PASSWORD, "POCKETBASE_PASSWORD is not set"
assert Settings.POCKETBASE_URL, "POCKETBASE_URL is not set"

In [ ]:
class Tweet(BaseModel):
    tweet_id: str
    text: str
    status_link: str
    user_id: str
    is_extremist: Optional[bool] = False
    is_annotated: Optional[bool] = False
    in_reply_to_status_link: Optional[str] = None
    in_reply_to_status_id: Optional[str]
    bookmark_count: int
    views: Optional[int]
    retweet_count: int
    favorite_count: int
    reply_count: int
    quote_count: int
    conversation_id: str
    retweet_status_id: Optional[str]
    quoted_status_id: Optional[str]
    community_note: Optional[str]
    language: str
    source: Optional[str]
    creation_date: str
    has_blm_hashtag: bool
    fetched_replies: Optional[bool] = False
    is_reply_to_blm: Optional[bool] = None
    is_hateful: Optional[Literal[0, 1, 2, ""]] = None

class User(BaseModel):
    user_id: str
    username: Optional[str]
    name: Optional[str]
    follower_count: int
    following_count: int
    favourites_count: int
    listed_count: int
    number_of_tweets: int
    is_private: Optional[bool]
    is_verified: Optional[bool]
    is_blue_verified: bool
    bot: bool
    location: Optional[str]
    description: str
    status: Literal["fetched", "not fetched", "fetching", ""] = "not fetched"
    creation_date: Optional[str]


In [ ]:
class TweetyScraper:
    def __init__(self, previous_session: bool = True):
        self.previous_session = previous_session

    async def login(self) -> TwitterAsync:
        app: TwitterAsync = TwitterAsync("session")
        if self.previous_session:
            await app.connect()
        else:
            USERNAME: str = Settings.X_USERNAME.value
            PASSWORD: str = Settings.X_PASSWORD.value
            TOTP: int = Settings.X_TOTP.value

            assert USERNAME and PASSWORD and TOTP, (
                "Username, password, and TOTP must be provided in the settings."
            )
            await app.sign_in(USERNAME, PASSWORD, extra=TOTP)

        return app

    async def get_tweets_of_user(self, username: str, pages: int = 100, wait_time: int = 30) -> list[dict]:
        app: TwitterAsync = await self.login()

        tweets: list[TweetyTweet] = await app.search(
            f"(from:{username})", wait_time=wait_time, pages=pages, filter_=SearchFilters.Latest()
        )


        if not tweets:
            logger.warning(f"No tweets found for user {username}.")
            return []

        if len(tweets) == 0:
            logger.warning(f"No tweets found for user {username}.")
            return []

        validated_tweets = []

        try:
            for tweet in tweets:
                if isinstance(tweet, TweetyTweet):
                    tweet_data = self.process_tweety_tweet(tweet)
                    validated_tweets.append(tweet_data)
                elif isinstance(tweet, SelfThread):
                    for t in tweet.tweets:
                        assert isinstance(t, TweetyTweet), (
                            f"Expected TweetyTweet type, got {type(t)}"
                        )
                        tweet_data = self.process_tweety_tweet(t)
                        validated_tweets.append(tweet_data)
                else:
                    logger.warning(f"Unknown tweet type: {type(tweet)}")
        except KeyboardInterrupt:
            logger.info("Scraping interrupted by user.")
            exit(0)
        except Exception as e:
            logger.error(f"Error processing tweets for user {username}: {e}")

        return validated_tweets

    @staticmethod
    def process_tweety_tweet(tweet: TweetyTweet) -> dict:
        search_terms = [
            "#blacklivesmatter",
            "#blm",
            "#blacklivesmatters",
        ]

        conversation_id = tweet.__dict__.get("_original_tweet", {}).get(
            "conversation_id_str", None
        )

        if not conversation_id:
            pprint(tweet.__dict__)
            raise ValueError(
                f"Conversation ID not found in tweet {tweet.id}. Please check the tweet structure."
            )

        views = tweet.views if isinstance(tweet.views, int) else None

        tweet = Tweet(
            tweet_id=tweet.id,
            text=tweet.text,
            status_link=tweet.url,
            user_id=tweet.author.id,
            in_reply_to_status_id=tweet.replied_to,
            bookmark_count=tweet.bookmark_count,
            views=views,
            retweet_count=tweet.retweet_counts,
            favorite_count=tweet.likes,
            reply_count=tweet.reply_counts,
            quote_count=tweet.quote_counts,
            conversation_id=conversation_id,
            retweet_status_id=tweet.retweeted_tweet.id
            if tweet.retweeted_tweet
            else None,
            quoted_status_id=tweet.quoted_tweet.id if tweet.quoted_tweet else None,
            community_note=tweet.community_note,
            language=tweet.language,
            source=tweet.source,
            creation_date=str(tweet.created_on),
            has_blm_hashtag=any(term in tweet.text.lower() for term in search_terms),
        )

        return tweet.model_dump()


In [ ]:
class PBWarehouse:
    def __init__(self, url: str = Settings.POCKETBASE_URL.value):
        self.client = PocketBase(url)
        self.authenticated = self.client.admins.auth_with_password(
            email=Settings.POCKETBASE_EMAIL.value, password=Settings.POCKETBASE_PASSWORD.value
        )

    def ingest_tweet(self, tweet: dict) -> dict[str, Record]:
        assert isinstance(tweet, dict), "Input must be a dictionary"
        processed_tweet = self._process_tweet(tweet)
        processed_user = self._process_user(tweet)

        record_tweet: Record = None
        record_user: Record = None

        try:
            record_tweet = self.client.collection("tweets_v2").create(processed_tweet)
            logger.success(
                f"Successfully created tweet record with ID: {record_tweet.id}"
            )
        except ClientResponseError as e:
            if "validation_not_unique" in str(e):
                logger.info(
                    f"Tweet with ID {processed_tweet['tweet_id']} already exists. Skipping."
                )

        try:
            record_user = self.client.collection("tweet_users").create(processed_user)
            logger.success(
                f"Successfully created user record with ID: {record_user.id}"
            )
        except ClientResponseError as e:
            if "validation_not_unique" in str(e):
                logger.info(
                    f"User with ID {processed_user['user_id']} already exists. Skipping."
                )
        return {
            "record_tweet": record_tweet,
            "record_user": record_user,
        }

    @staticmethod
    def _process_tweet(tweet: dict) -> dict[str, any]:
        assert isinstance(tweet, dict), "Input must be a dictionary"

        text = tweet.get("text", "")
        user_id = tweet.get("user", {}).get("user_id", "")
        username = tweet.get("user", {}).get("username", "")
        status_link = f"https://x.com/{username}/status/{tweet.get('tweet_id', '')}"
        retweet_status_id = tweet.get("retweet_tweet_id", {})
        quoted_status_id = tweet.get("quoted_status_id", {})
        community_note = tweet.get("community_note", {})
        search_terms = [
            "#blacklivesmatter",
            "#blm",
            "#blacklivesmatters",
        ]

        tweet["has_blm_hashtag"] = any(term in text.lower() for term in search_terms)
        tweet["user_id"] = user_id
        tweet["status_link"] = status_link
        tweet["retweet_status_id"] = retweet_status_id
        tweet["quoted_status_id"] = quoted_status_id
        tweet["community_note"] = (
            community_note.get("subtitle", {}).get("text", "")
            if community_note
            else None
        )

        parsed_tweet = Tweet(**tweet)
        return parsed_tweet.model_dump()

    @staticmethod
    def _process_user(tweet: dict) -> dict[str, any]:
        assert isinstance(tweet, dict), "Input must be a dictionary"
        user = tweet.get("user", {})
        parsed_user = User(**user)
        return parsed_user.model_dump()

    def update_has_fetched_replies(self, tweet_id: str) -> Record:
        assert isinstance(tweet_id, str), "tweet_id must be a string"
        record = self.client.collection("tweets_v2").get_list(
            1, 1, {"filter": f"tweet_id = '{tweet_id}'"}
        )

        updated_record = self.client.collection("tweets_v2").update(
            record.items[0].id, {"fetched_replies": True}
        )
        return updated_record

    def get_user_by_id(self, user_id: str) -> Record:
        assert isinstance(user_id, str), "user_id must be a string"
        user = self.client.collection("tweet_users").get_first_list_item(
            f"user_id = '{user_id}'"
        )
        return user if user else None

    def get_tweet_with_no_classification(self) -> Record:
        """Get a tweet that has not been classified yet."""
        tweetRecord = self.client.collection("tweets_v2").get_first_list_item(
            "is_hateful = NULL",
        )
        return tweetRecord if tweetRecord else None

    def get_user_with_not_fetched_tweets(self) -> Record:
        """Get a user that has 'not fetched' tweets yet."""
        userRecord = self.client.collection("users_tweets_status").get_first_list_item(
            "status = 'not fetched' || status = NULL",
        )
        userRecord = self.get_user_by_id(userRecord.user_id)

        return userRecord if userRecord else None


In [ ]:
def ensure_path(path: Path) -> None:
    """
    Ensure that the given path is a Path object.

    Args:
        path (str): The path to be converted.

    Returns:
        Path: A Path object representing the given path.
    """
    assert isinstance(path, Path), f"Expected Path type, got {type(path)}"

    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)
        print(f"Created directory: {path}")

    return path


The files will be saved in the 'tweety-capstone-2' folder in your Google Drive

In [ ]:
# Create the folder if it doesn't exist
SAVE_DIR = ensure_path(DRIVE / "tweety-capstone-2")

In [ ]:
async def get_all_users_tweets_by_tweety(
    max_requests: int | None = None, wait_time: int = 30
) -> None:
    pb = PBWarehouse()
    scraper = TweetyScraper(
        previous_session=False
    )
    TWEETS_PER_PAGE = 20

    have_data = True
    while have_data:
        try:
            userRecord: Record = pb.get_user_with_not_fetched_tweets()
            user: User = User(**userRecord.__dict__)
            pages = math.ceil(user.number_of_tweets / TWEETS_PER_PAGE)

            pb.client.collection("tweet_users").update(
                userRecord.id, {"status": "fetching"}
            )
            logger.info(
                f"Fetching tweets for user: {user.username} "
                f"(ID: {user.user_id}), tweets: {user.number_of_tweets}, pages: {pages}."
            )

            # Fetch tweets for the user
            tweets: list[dict] = await scraper.get_tweets_of_user(
                                            username=user.username,
                                            pages=max_requests if max_requests else pages,
                                            wait_time=wait_time,
                                      )

            if not tweets:
                logger.warning(f"No tweets found for user {user.username}.")
                pb.client.collection("tweet_users").update(
                    userRecord.id, {"status": "fetched"}
                )
                continue

            logger.info(
                f"Fetched {len(tweets)} tweets for user {user.username} (ID: {user.user_id})"
            )

            # Save tweets to JSON files
            json_filename = SAVE_DIR / f"{user.username}.json"
            with open(json_filename, "w", encoding="utf-8") as f:
                json.dump(tweets, f, ensure_ascii=False, indent=4)

            pb.client.collection("tweet_users").update(
                userRecord.id,
                {
                    "status": "fetched",
                },
            )
            logger.info(f"Updated user {user.username} with {len(tweets)} tweets.")
        except ClientResponseError as e:
            if "The requested resource wasn't found." in str(e):
                logger.info("No more users to fetch tweets for.")
                pb.client.collection("tweet_users").update(
                    userRecord.id, {"status": "not fetched"}
                )
                have_data = False
            else:
                logger.error(f"ClientResponseError: {e}")
                pb.client.collection("tweet_users").update(
                    userRecord.id, {"status": "not fetched"}
                )
                have_data = False
        except Exception as e:
            logger.error(f"Error fetching tweets: {type(e).__name__} - {e}")
            have_data = False
            pb.client.collection("tweet_users").update(
                userRecord.id, {"status": "not fetched"}
            )



## Getting the data

Will save in the 'tweety' folder in you Google Drive.

In [ ]:
await get_all_users_tweets_by_tweety()

## Inserting the data

The files in the 'tweety-capstone-2' folder will be saved in the data warehouse (Pocketbase).

In [ ]:
def ingest_tweety_tweets() -> None:
    """Ingest tweets from Tweety into the PocketBase warehouse."""
    pb = PBWarehouse()

    for user_tweets_file in SAVE_DIR.iterdir():
        if user_tweets_file.suffix != ".json":
            logger.warning(f"Skipping non-JSON file: {user_tweets_file.name}")
            continue

        try:
            with open(user_tweets_file, "r", encoding="utf-8") as f:
                tweets = json.load(f)

            assert isinstance(tweets, list), (
                f"Tweets data must be a list, got {type(tweets)}"
            )

            for tweet in tweets:
                assert isinstance(tweet, dict), (
                    f"Each tweet must be a dictionary, got {type(tweet)}"
                )
                assert "tweet_id" in tweet, "Tweet data must contain 'tweet_id'"
                pb.client.collection("tweets_v2").create(tweet)

            logger.success(f"Successfully ingested tweets from {user_tweets_file.name}")

        except ClientResponseError as e:
            if "validation_not_unique" in str(e):
                logger.info(
                    f"Tweet with ID {tweet['tweet_id']} already exists. Skipping."
                )
                continue
            else:
                logger.error(
                    f"ClientResponseError while ingesting {user_tweets_file.name}: {e}"
                )
        except Exception as e:
            logger.error(
                f"Error ingesting {user_tweets_file.name}: {type(e).__name__} - {e}"
            )

In [ ]:
ingest_tweety_tweets()